# PS S6E4 - exp_20260410_029_ridge_alpha_refine_topstacks

目的:
 - 028 の screening で有望だった stack 候補について、
   ridge alpha を二段階で詰めて本番寄りに比較する

対象:
 - main: 018 + 025 + 026
 - compare: 018 + 024 + 026
 - baseline: 025 単体

方針:
 - simple blend は今回やらない
 - ridge のみ
 - coarse -> local の二段階で alpha を探索
 - raw CV / biased CV / 提出LB で判断する

## Imports

In [1]:
import os
import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.metrics import balanced_accuracy_score

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 200)

## CFG

In [2]:
class CFG:
    EXP_ID = "exp_20260410_029_ridge_alpha_refine_topstacks"
    EXP_NAME = "ridge_alpha_refine_topstacks"
    SAVE_DIR = Path(f"/kaggle/working/{EXP_ID}")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
    SAMPLE_SUB_PATH = "/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv"
    NPY_PATH = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/"

    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"
    SEED = 42
    N_CLASSES = 3

    COARSE_ALPHA_GRID = [
        1e-4, 3e-4,
        1e-3, 3e-3,
        1e-2, 3e-2,
        1e-1, 3e-1,
        1.0, 3.0,
        10.0, 30.0, 100.0,
    ]

    LOCAL_ALPHA_GRID_MAP = {
        1e-4: [1e-4, 2e-4, 3e-4, 5e-4, 7e-4, 1e-3],
        3e-4: [1e-4, 2e-4, 3e-4, 5e-4, 7e-4, 1e-3],
        1e-3: [3e-4, 5e-4, 7e-4, 1e-3, 2e-3, 3e-3],
        3e-3: [1e-3, 2e-3, 3e-3, 5e-3, 7e-3, 1e-2],
        1e-2: [3e-3, 5e-3, 7e-3, 1e-2, 2e-2, 3e-2],
        3e-2: [1e-2, 2e-2, 3e-2, 5e-2, 7e-2, 1e-1],
        1e-1: [3e-2, 5e-2, 7e-2, 1e-1, 1.5e-1, 2e-1, 3e-1],
        3e-1: [1e-1, 1.5e-1, 2e-1, 3e-1, 5e-1, 7e-1, 1.0],
        1.0: [3e-1, 5e-1, 7e-1, 1.0, 1.5, 2.0, 3.0],
        3.0: [1.0, 1.5, 2.0, 3.0, 5.0, 7.0, 10.0],
        10.0: [3.0, 5.0, 7.0, 10.0, 15.0, 20.0, 30.0],
        30.0: [10.0, 15.0, 20.0, 30.0, 50.0, 70.0, 100.0],
        100.0: [30.0, 50.0, 70.0, 100.0, 150.0, 200.0, 300.0],
    }

    BIAS_GRID = [
        [1.0, 1.0, 1.0],
        [1.2, 1.2, 1.2],
        [1.5, 1.5, 1.5],
        [1.5, 1.3, 1.8],
        [1.7, 1.5, 2.3],
        [1.8, 1.5, 2.4],
    ]

## utility

In [3]:
def seed_everything(seed):
    np.random.seed(seed)
    random.seed(seed)

def normalize_proba(p):
    p = np.asarray(p, dtype=float)
    p = np.clip(p, 1e-15, None)
    row_sum = p.sum(axis=1, keepdims=True)
    row_sum = np.where(row_sum == 0, 1.0, row_sum)
    return p / row_sum

def balanced_acc_score_mc(y_true, proba_or_pred):
    arr = np.asarray(proba_or_pred)
    if arr.ndim == 2:
        pred = np.argmax(arr, axis=1)
    else:
        pred = arr
    return float(balanced_accuracy_score(y_true, pred))

def one_hot(y, n_classes):
    y = np.asarray(y).astype(int)
    out = np.zeros((len(y), n_classes), dtype=float)
    out[np.arange(len(y)), y] = 1.0
    return out

def apply_class_weights_to_proba(proba, class_weights):
    class_weights = np.asarray(class_weights, dtype=float)
    adjusted = proba * class_weights[None, :]
    return normalize_proba(adjusted)

seed_everything(CFG.SEED)

## load target / submission template

In [4]:
train_df = pd.read_csv(CFG.TRAIN_PATH)
test_df = pd.read_csv(CFG.TEST_PATH)
sample_sub = pd.read_csv(CFG.SAMPLE_SUB_PATH)

target2idx = {v: i for i, v in enumerate(train_df[CFG.TARGET_COL].unique())}
idx2target = {v: k for k, v in target2idx.items()}
y = train_df[CFG.TARGET_COL].map(target2idx).values.astype(int)

print(train_df.shape, test_df.shape)
print(target2idx)

(630000, 21) (270000, 20)
{'Low': 0, 'Medium': 1, 'High': 2}


## load model npy

In [5]:
model_dict = {
    "018": (
        np.load(CFG.NPY_PATH + "oof_cat_pairwise_te_bias_from_shared_min_proba_biased.npy"),
        np.load(CFG.NPY_PATH + "pred_cat_pairwise_te_bias_from_shared_min_proba_biased.npy"),
    ),
    "024": (
        np.load(CFG.NPY_PATH + "oof_xgb_digit_orderedte_min_proba_biased.npy"),
        np.load(CFG.NPY_PATH + "pred_xgb_digit_orderedte_min_proba_biased.npy"),
    ),
    "025": (
        np.load(CFG.NPY_PATH + "oof_lgb_digit_te_min_proba_biased.npy"),
        np.load(CFG.NPY_PATH + "pred_lgb_digit_te_min_proba_biased.npy"),
    ),
    "026": (
        np.load(CFG.NPY_PATH + "oof_realmlp_pair_te_min_proba_biased.npy"),
        np.load(CFG.NPY_PATH + "pred_realmlp_pair_te_min_proba_biased.npy"),
    ),
}

## class order fix

In [6]:
perm = [2, 0, 1]

for name in ["018", "024", "025", "026"]:
    oof_t, pred_t = model_dict[name]
    model_dict[name] = (oof_t[:, perm], pred_t[:, perm])

for name, (oof_arr, pred_arr) in model_dict.items():
    print(name, oof_arr.shape, pred_arr.shape, np.round(oof_arr[:3].sum(axis=1), 6))

018 (630000, 3) (270000, 3) [1. 1. 1.]
024 (630000, 3) (270000, 3) [1. 1. 1.]
025 (630000, 3) (270000, 3) [1. 1. 1.]
026 (630000, 3) (270000, 3) [1. 1. 1.]


## target candidate sets

In [7]:
candidate_sets = {
    "baseline_025": ["025"],
    "main_018_025_026": ["018", "025", "026"],
    "compare_018_024_026": ["018", "024", "026"],
}

candidate_sets

{'baseline_025': ['025'],
 'main_018_025_026': ['018', '025', '026'],
 'compare_018_024_026': ['018', '024', '026']}

## ridge helpers

In [8]:
def build_meta_features(model_names, model_dict, use_logit=False):
    oof_feats = []
    pred_feats = []

    for m in model_names:
        oof_p = model_dict[m][0].copy()
        pred_p = model_dict[m][1].copy()

        if use_logit:
            eps = 1e-15
            oof_p = np.log(np.clip(oof_p, eps, 1 - eps) / np.clip(1 - oof_p, eps, 1 - eps))
            pred_p = np.log(np.clip(pred_p, eps, 1 - eps) / np.clip(1 - pred_p, eps, 1 - eps))

        oof_feats.append(oof_p)
        pred_feats.append(pred_p)

    X_oof = np.concatenate(oof_feats, axis=1)
    X_pred = np.concatenate(pred_feats, axis=1)
    return X_oof, X_pred


def ridge_multiclass_blend(model_names, model_dict, y_true, alpha=1.0, use_logit=False):
    if len(model_names) == 1:
        return model_dict[model_names[0]][0].copy(), model_dict[model_names[0]][1].copy()

    X_oof, X_pred = build_meta_features(model_names, model_dict, use_logit=use_logit)
    y_onehot = one_hot(y_true, CFG.N_CLASSES)

    pred_oof = np.zeros((len(y_true), CFG.N_CLASSES), dtype=float)
    pred_test = np.zeros((X_pred.shape[0], CFG.N_CLASSES), dtype=float)

    for c in range(CFG.N_CLASSES):
        reg = Ridge(alpha=alpha, random_state=CFG.SEED)
        reg.fit(X_oof, y_onehot[:, c])
        pred_oof[:, c] = reg.predict(X_oof)
        pred_test[:, c] = reg.predict(X_pred)

    pred_oof = normalize_proba(pred_oof)
    pred_test = normalize_proba(pred_test)
    return pred_oof, pred_test


def search_best_bias_grid(oof_proba, y_true, bias_grid):
    best_score = balanced_acc_score_mc(y_true, oof_proba)
    best_weights = np.ones(CFG.N_CLASSES, dtype=float)

    for cw in bias_grid:
        cw = np.asarray(cw, dtype=float)
        adjusted = apply_class_weights_to_proba(oof_proba, cw)
        score = balanced_acc_score_mc(y_true, adjusted)
        if score > best_score:
            best_score = score
            best_weights = cw.copy()

    return best_weights, best_score

## coarse alpha search

In [9]:
coarse_results = []

for set_name, model_names in candidate_sets.items():
    print("=" * 100)
    print("COARSE SEARCH:", set_name, model_names)
    print("=" * 100)

    # baseline 025 は alpha 不要
    if len(model_names) == 1:
        oof_blend, pred_blend = ridge_multiclass_blend(model_names, model_dict, y, alpha=1.0, use_logit=False)
        raw_cv = balanced_acc_score_mc(y, oof_blend)
        best_w, biased_cv = search_best_bias_grid(oof_blend, y, CFG.BIAS_GRID)

        coarse_results.append({
            "set_name": set_name,
            "models": "+".join(model_names),
            "alpha": None,
            "raw_cv_ba": raw_cv,
            "biased_cv_ba": biased_cv,
            "best_class_weights": best_w.tolist(),
        })

        print(f"baseline raw={raw_cv:.6f} | biased={biased_cv:.6f}")
        continue

    for alpha in CFG.COARSE_ALPHA_GRID:
        oof_blend, pred_blend = ridge_multiclass_blend(model_names, model_dict, y, alpha=alpha, use_logit=False)
        raw_cv = balanced_acc_score_mc(y, oof_blend)
        best_w, biased_cv = search_best_bias_grid(oof_blend, y, CFG.BIAS_GRID)

        coarse_results.append({
            "set_name": set_name,
            "models": "+".join(model_names),
            "alpha": alpha,
            "raw_cv_ba": raw_cv,
            "biased_cv_ba": biased_cv,
            "best_class_weights": best_w.tolist(),
        })

    tmp = pd.DataFrame([r for r in coarse_results if r["set_name"] == set_name])
    display(tmp.sort_values("biased_cv_ba", ascending=False).head(10))

COARSE SEARCH: baseline_025 ['025']
baseline raw=0.014638 | biased=0.015312
COARSE SEARCH: main_018_025_026 ['018', '025', '026']


,set_name,models,alpha,raw_cv_ba,biased_cv_ba,best_class_weights
10,main_018_025_026,018+025+026,10.0000,0.979633,0.980131,"[1.5, 1.3, 1.8]"
1,main_018_025_026,018+025+026,0.0003,0.979639,0.980126,"[1.5, 1.3, 1.8]"
2,main_018_025_026,018+025+026,0.0010,0.979639,0.980126,"[1.5, 1.3, 1.8]"
3,main_018_025_026,018+025+026,0.0030,0.979639,0.980126,"[1.5, 1.3, 1.8]"
0,main_018_025_026,018+025+026,0.0001,0.979639,0.980126,"[1.5, 1.3, 1.8]"
4,main_018_025_026,018+025+026,0.0100,0.979639,0.980126,"[1.5, 1.3, 1.8]"
5,main_018_025_026,018+025+026,0.0300,0.979639,0.980126,"[1.5, 1.3, 1.8]"
7,main_018_025_026,018+025+026,0.3000,0.979639,0.980126,"[1.5, 1.3, 1.8]"
6,main_018_025_026,018+025+026,0.1000,0.979639,0.980126,"[1.5, 1.3, 1.8]"
8,main_018_025_026,018+025+026,1.0000,0.979638,0.980126,"[1.5, 1.3, 1.8]"


COARSE SEARCH: compare_018_024_026 ['018', '024', '026']


,set_name,models,alpha,raw_cv_ba,biased_cv_ba,best_class_weights
11,compare_018_024_026,018+024+026,30.0000,0.979510,0.979997,"[1.8, 1.5, 2.4]"
9,compare_018_024_026,018+024+026,3.0000,0.979486,0.979978,"[1.8, 1.5, 2.4]"
8,compare_018_024_026,018+024+026,1.0000,0.979484,0.979974,"[1.8, 1.5, 2.4]"
6,compare_018_024_026,018+024+026,0.1000,0.979484,0.979973,"[1.8, 1.5, 2.4]"
7,compare_018_024_026,018+024+026,0.3000,0.979485,0.979973,"[1.8, 1.5, 2.4]"
12,compare_018_024_026,018+024+026,100.0000,0.979612,0.979965,"[1.8, 1.5, 2.4]"
10,compare_018_024_026,018+024+026,10.0000,0.979484,0.979965,"[1.8, 1.5, 2.4]"
4,compare_018_024_026,018+024+026,0.0100,0.979484,0.979956,"[1.8, 1.5, 2.4]"
1,compare_018_024_026,018+024+026,0.0003,0.979484,0.979956,"[1.8, 1.5, 2.4]"
0,compare_018_024_026,018+024+026,0.0001,0.979484,0.979956,"[1.8, 1.5, 2.4]"


## coarse results table

In [10]:
coarse_df = pd.DataFrame(coarse_results)
coarse_df = coarse_df.sort_values(["set_name", "biased_cv_ba", "raw_cv_ba"], ascending=[True, False, False]).reset_index(drop=True)
display(coarse_df)

,set_name,models,alpha,raw_cv_ba,biased_cv_ba,best_class_weights
0,baseline_025,025,NaN,0.014638,0.015312,"[1.7, 1.5, 2.3]"
1,compare_018_024_026,018+024+026,30.0000,0.979510,0.979997,"[1.8, 1.5, 2.4]"
2,compare_018_024_026,018+024+026,3.0000,0.979486,0.979978,"[1.8, 1.5, 2.4]"
3,compare_018_024_026,018+024+026,1.0000,0.979484,0.979974,"[1.8, 1.5, 2.4]"
4,compare_018_024_026,018+024+026,0.3000,0.979485,0.979973,"[1.8, 1.5, 2.4]"
5,compare_018_024_026,018+024+026,0.1000,0.979484,0.979973,"[1.8, 1.5, 2.4]"
6,compare_018_024_026,018+024+026,100.0000,0.979612,0.979965,"[1.8, 1.5, 2.4]"
7,compare_018_024_026,018+024+026,10.0000,0.979484,0.979965,"[1.8, 1.5, 2.4]"
8,compare_018_024_026,018+024+026,0.0001,0.979484,0.979956,"[1.8, 1.5, 2.4]"
9,compare_018_024_026,018+024+026,0.0003,0.979484,0.979956,"[1.8, 1.5, 2.4]"


## pick coarse best alpha

In [11]:
coarse_best = (
    coarse_df.sort_values(["set_name", "biased_cv_ba", "raw_cv_ba"], ascending=[True, False, False])
    .groupby("set_name", as_index=False)
    .first()
)

display(coarse_best)

,set_name,models,alpha,raw_cv_ba,biased_cv_ba,best_class_weights
0,baseline_025,025,NaN,0.014638,0.015312,"[1.7, 1.5, 2.3]"
1,compare_018_024_026,018+024+026,30.0,0.979510,0.979997,"[1.8, 1.5, 2.4]"
2,main_018_025_026,018+025+026,10.0,0.979633,0.980131,"[1.5, 1.3, 1.8]"


## local alpha search

In [12]:
local_results = []

for _, row in coarse_best.iterrows():
    set_name = row["set_name"]
    model_names = candidate_sets[set_name]

    if len(model_names) == 1:
        local_results.append({
            "set_name": set_name,
            "models": "+".join(model_names),
            "alpha": None,
            "raw_cv_ba": row["raw_cv_ba"],
            "biased_cv_ba": row["biased_cv_ba"],
            "best_class_weights": row["best_class_weights"],
        })
        continue

    best_alpha_from_coarse = float(row["alpha"])
    local_grid = CFG.LOCAL_ALPHA_GRID_MAP[best_alpha_from_coarse]

    print("=" * 100)
    print("LOCAL SEARCH:", set_name, model_names, "coarse best =", best_alpha_from_coarse)
    print("local grid =", local_grid)
    print("=" * 100)

    for alpha in local_grid:
        oof_blend, pred_blend = ridge_multiclass_blend(model_names, model_dict, y, alpha=alpha, use_logit=False)
        raw_cv = balanced_acc_score_mc(y, oof_blend)
        best_w, biased_cv = search_best_bias_grid(oof_blend, y, CFG.BIAS_GRID)

        local_results.append({
            "set_name": set_name,
            "models": "+".join(model_names),
            "alpha": alpha,
            "raw_cv_ba": raw_cv,
            "biased_cv_ba": biased_cv,
            "best_class_weights": best_w.tolist(),
        })

    tmp = pd.DataFrame([r for r in local_results if r["set_name"] == set_name])
    display(tmp.sort_values("biased_cv_ba", ascending=False).head(10))

LOCAL SEARCH: compare_018_024_026 ['018', '024', '026'] coarse best = 30.0
local grid = [10.0, 15.0, 20.0, 30.0, 50.0, 70.0, 100.0]


,set_name,models,alpha,raw_cv_ba,biased_cv_ba,best_class_weights
3,compare_018_024_026,018+024+026,30.0,0.979510,0.979997,"[1.8, 1.5, 2.4]"
6,compare_018_024_026,018+024+026,100.0,0.979612,0.979965,"[1.8, 1.5, 2.4]"
0,compare_018_024_026,018+024+026,10.0,0.979484,0.979965,"[1.8, 1.5, 2.4]"
5,compare_018_024_026,018+024+026,70.0,0.979573,0.979964,"[1.7, 1.5, 2.3]"
1,compare_018_024_026,018+024+026,15.0,0.979488,0.979962,"[1.8, 1.5, 2.4]"
2,compare_018_024_026,018+024+026,20.0,0.979494,0.979958,"[1.8, 1.5, 2.4]"
4,compare_018_024_026,018+024+026,50.0,0.979545,0.979953,"[1.8, 1.5, 2.4]"


LOCAL SEARCH: main_018_025_026 ['018', '025', '026'] coarse best = 10.0
local grid = [3.0, 5.0, 7.0, 10.0, 15.0, 20.0, 30.0]


,set_name,models,alpha,raw_cv_ba,biased_cv_ba,best_class_weights
3,main_018_025_026,018+025+026,10.0,0.979633,0.980131,"[1.5, 1.3, 1.8]"
4,main_018_025_026,018+025+026,15.0,0.979639,0.980129,"[1.5, 1.3, 1.8]"
5,main_018_025_026,018+025+026,20.0,0.979641,0.980129,"[1.5, 1.3, 1.8]"
2,main_018_025_026,018+025+026,7.0,0.979635,0.980127,"[1.5, 1.3, 1.8]"
0,main_018_025_026,018+025+026,3.0,0.979635,0.980126,"[1.5, 1.3, 1.8]"
1,main_018_025_026,018+025+026,5.0,0.979637,0.980126,"[1.5, 1.3, 1.8]"
6,main_018_025_026,018+025+026,30.0,0.979640,0.980125,"[1.5, 1.3, 1.8]"


## local results table

In [13]:
local_df = pd.DataFrame(local_results)
local_df = local_df.sort_values(["set_name", "biased_cv_ba", "raw_cv_ba"], ascending=[True, False, False]).reset_index(drop=True)
display(local_df)

,set_name,models,alpha,raw_cv_ba,biased_cv_ba,best_class_weights
0,baseline_025,025,NaN,0.014638,0.015312,"[1.7, 1.5, 2.3]"
1,compare_018_024_026,018+024+026,30.0,0.979510,0.979997,"[1.8, 1.5, 2.4]"
2,compare_018_024_026,018+024+026,100.0,0.979612,0.979965,"[1.8, 1.5, 2.4]"
3,compare_018_024_026,018+024+026,10.0,0.979484,0.979965,"[1.8, 1.5, 2.4]"
4,compare_018_024_026,018+024+026,70.0,0.979573,0.979964,"[1.7, 1.5, 2.3]"
5,compare_018_024_026,018+024+026,15.0,0.979488,0.979962,"[1.8, 1.5, 2.4]"
6,compare_018_024_026,018+024+026,20.0,0.979494,0.979958,"[1.8, 1.5, 2.4]"
7,compare_018_024_026,018+024+026,50.0,0.979545,0.979953,"[1.8, 1.5, 2.4]"
8,main_018_025_026,018+025+026,10.0,0.979633,0.980131,"[1.5, 1.3, 1.8]"
9,main_018_025_026,018+025+026,20.0,0.979641,0.980129,"[1.5, 1.3, 1.8]"


## final best per set

In [14]:
final_best = (
    local_df.sort_values(["set_name", "biased_cv_ba", "raw_cv_ba"], ascending=[True, False, False])
    .groupby("set_name", as_index=False)
    .first()
)

display(final_best)

,set_name,models,alpha,raw_cv_ba,biased_cv_ba,best_class_weights
0,baseline_025,025,NaN,0.014638,0.015312,"[1.7, 1.5, 2.3]"
1,compare_018_024_026,018+024+026,30.0,0.979510,0.979997,"[1.8, 1.5, 2.4]"
2,main_018_025_026,018+025+026,10.0,0.979633,0.980131,"[1.5, 1.3, 1.8]"


## rebuild final best blends

In [15]:
final_outputs = {}
summary_rows = []

for _, row in final_best.iterrows():
    set_name = row["set_name"]
    model_names = candidate_sets[set_name]
    alpha = row["alpha"]

    if len(model_names) == 1:
        oof_blend, pred_blend = ridge_multiclass_blend(model_names, model_dict, y, alpha=1.0, use_logit=False)
    else:
        oof_blend, pred_blend = ridge_multiclass_blend(model_names, model_dict, y, alpha=float(alpha), use_logit=False)

    best_w = np.asarray(row["best_class_weights"], dtype=float)
    oof_biased = apply_class_weights_to_proba(oof_blend, best_w)
    pred_biased = apply_class_weights_to_proba(pred_blend, best_w)

    final_outputs[set_name] = {
        "models": model_names,
        "alpha": None if pd.isna(alpha) else float(alpha),
        "oof_raw": oof_blend,
        "pred_raw": pred_blend,
        "oof_biased": oof_biased,
        "pred_biased": pred_biased,
        "best_class_weights": best_w.tolist(),
    }

    summary_rows.append({
        "set_name": set_name,
        "models": "+".join(model_names),
        "alpha": None if pd.isna(alpha) else float(alpha),
        "raw_cv_ba": balanced_acc_score_mc(y, oof_blend),
        "biased_cv_ba": balanced_acc_score_mc(y, oof_biased),
        "best_class_weights": best_w.tolist(),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("biased_cv_ba", ascending=False).reset_index(drop=True)
display(summary_df)

,set_name,models,alpha,raw_cv_ba,biased_cv_ba,best_class_weights
0,main_018_025_026,018+025+026,10.0,0.979633,0.980131,"[1.5, 1.3, 1.8]"
1,compare_018_024_026,018+024+026,30.0,0.979510,0.979997,"[1.8, 1.5, 2.4]"
2,baseline_025,025,NaN,0.014638,0.015312,"[1.7, 1.5, 2.3]"


## save summary tables

In [16]:
coarse_df.to_csv(CFG.SAVE_DIR / "coarse_alpha_results.csv", index=False)
local_df.to_csv(CFG.SAVE_DIR / "local_alpha_results.csv", index=False)
summary_df.to_csv(CFG.SAVE_DIR / "final_summary_results.csv", index=False)

with open(CFG.SAVE_DIR / "coarse_alpha_results.json", "w", encoding="utf-8") as f:
    json.dump(coarse_results, f, ensure_ascii=False, indent=2)

with open(CFG.SAVE_DIR / "local_alpha_results.json", "w", encoding="utf-8") as f:
    json.dump(local_results, f, ensure_ascii=False, indent=2)

## save best overall artifacts

In [17]:
best_row = summary_df.iloc[0]
best_set = best_row["set_name"]
best_obj = final_outputs[best_set]

print("best_set =", best_set)
print("best_models =", best_obj["models"])
print("best_alpha =", best_obj["alpha"])
print("best_class_weights =", best_obj["best_class_weights"])

np.save(CFG.SAVE_DIR / f"oof_{best_set}_ridge_refined_proba.npy", best_obj["oof_raw"])
np.save(CFG.SAVE_DIR / f"pred_{best_set}_ridge_refined_proba.npy", best_obj["pred_raw"])
np.save(CFG.SAVE_DIR / f"oof_{best_set}_ridge_refined_proba_biased.npy", best_obj["oof_biased"])
np.save(CFG.SAVE_DIR / f"pred_{best_set}_ridge_refined_proba_biased.npy", best_obj["pred_biased"])

best_pred_idx = np.argmax(best_obj["pred_biased"], axis=1)
submission = sample_sub.copy()
submission[CFG.TARGET_COL] = pd.Series(best_pred_idx).map(idx2target)
submission.to_csv(CFG.SAVE_DIR / f"submission_{best_set}_ridge_refined.csv", index=False)

display(submission.head())

best_set = main_018_025_026
best_models = ['018', '025', '026']
best_alpha = 10.0
best_class_weights = [1.5, 1.3, 1.8]


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


## save meta note

In [18]:
meta_note = {
    "candidate_sets": candidate_sets,
    "coarse_alpha_grid": CFG.COARSE_ALPHA_GRID,
    "local_alpha_grid_map": CFG.LOCAL_ALPHA_GRID_MAP,
    "bias_grid": CFG.BIAS_GRID,
    "best_set": best_set,
    "best_models": best_obj["models"],
    "best_alpha": best_obj["alpha"],
    "best_class_weights": best_obj["best_class_weights"],
}

with open(CFG.SAVE_DIR / "meta_note.json", "w", encoding="utf-8") as f:
    json.dump(meta_note, f, ensure_ascii=False, indent=2)

print("saved to:", CFG.SAVE_DIR)

saved to: /kaggle/working/exp_20260410_029_ridge_alpha_refine_topstacks


## quick check

In [19]:
display(summary_df)
print("done")

,set_name,models,alpha,raw_cv_ba,biased_cv_ba,best_class_weights
0,main_018_025_026,018+025+026,10.0,0.979633,0.980131,"[1.5, 1.3, 1.8]"
1,compare_018_024_026,018+024+026,30.0,0.979510,0.979997,"[1.8, 1.5, 2.4]"
2,baseline_025,025,NaN,0.014638,0.015312,"[1.7, 1.5, 2.3]"


done
